# 02 — Model Training, Comparison & Ensemble
## NIDS — Network Intrusion Detection System
**Goal:** Train multiple ML models, compare their performance, build an ensemble, and save the best model.

### Models we will train:
1. Logistic Regression (baseline — simple)
2. Decision Tree
3. Random Forest
4. XGBoost
5. **Voting Ensemble** (RF + XGBoost)
6. **Stacking Ensemble** (RF + XGBoost → Logistic Regression meta-model)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import joblib
import os
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.preprocessing   import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                      recall_score, classification_report, confusion_matrix,
                                      ConfusionMatrixDisplay)
from imblearn.over_sampling  import SMOTE
from xgboost                 import XGBClassifier

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('../data/processed', exist_ok=True)

print('All libraries imported!')


## Step 1 — Load & Clean Data


In [ ]:
DATA_PATH = '../data/raw/cicids2017_cleaned.csv'

print('Loading dataset...')
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

# Find label column
label_col = next((c for c in df.columns if 'label' in c.lower()), None)
print(f'Label column: "{label_col}"')

# Clean: remove NaN, Inf, duplicates
original_len = len(df)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

removed = original_len - len(df)
print(f'Removed {removed:,} bad rows. Remaining: {len(df):,}')
print(f'Class distribution:')
print(df[label_col].value_counts().to_string())


## Step 2 — Preprocessing


In [ ]:
# Separate features and labels
X = df.drop(columns=[label_col])
y = df[label_col]

# Drop non-numeric columns
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    X.drop(columns=non_numeric, inplace=True)

print(f'Features shape: {X.shape}')
print(f'Label classes: {y.nunique()}')


In [ ]:
# Label encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('Label encoding map:')
for i, cls in enumerate(le.classes_):
    count = (y_encoded == i).sum()
    print(f'  {i:2d}  →  {cls:<40}  ({count:,} samples)')

# Save encoder
joblib.dump(le, '../label_encoder.pkl')
print('\nLabel encoder saved → ../label_encoder.pkl')


In [ ]:
# Train / Test split (stratified 80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# Save test data for evaluate.py
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_test.npy', y_test)
print('Test data saved to data/processed/')


In [ ]:
# SMOTE — oversample minority classes (only on training data!)
print('Applying SMOTE...')
print(f'Before SMOTE — training samples: {len(X_train):,}')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f'After  SMOTE — training samples: {len(X_train_bal):,}')
print('New class distribution (balanced):')
unique, counts = np.unique(y_train_bal, return_counts=True)
for cls_id, cnt in zip(unique, counts):
    print(f'  {le.classes_[cls_id]:<40}  {cnt:,}')


In [ ]:
# StandardScaler — normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)   # fit on train only
X_test_scaled  = scaler.transform(X_test)             # apply same to test

joblib.dump(scaler, '../scaler.pkl')
print('Scaler fitted and saved → ../scaler.pkl')
print(f'Features: mean ≈ {X_train_scaled.mean():.4f}, std ≈ {X_train_scaled.std():.4f}')
print('(Should be close to 0 and 1 respectively)')


## Step 3 — Train & Evaluate All Models


In [ ]:
# Helper function to train and evaluate any model
def train_and_evaluate(model, name, X_tr, y_tr, X_te, y_te):
    """
    Train a model, compute all metrics, and return results as a dict.
    """
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred = model.predict(X_te)

    accuracy  = accuracy_score(y_te, y_pred)
    f1_macro  = f1_score(y_te, y_pred, average='macro',    zero_division=0)
    f1_weight = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    precision = precision_score(y_te, y_pred, average='macro', zero_division=0)
    recall    = recall_score(y_te, y_pred, average='macro',    zero_division=0)

    print(f'  Done in {train_time:.1f}s  |  '
          f'Accuracy={accuracy*100:.2f}%  |  Macro F1={f1_macro*100:.2f}%')

    return {
        'Model':      name,
        'Accuracy':   round(accuracy  * 100, 2),
        'Macro F1':   round(f1_macro  * 100, 2),
        'Weighted F1':round(f1_weight * 100, 2),
        'Precision':  round(precision * 100, 2),
        'Recall':     round(recall    * 100, 2),
        'Train Time': f'{train_time:.1f}s',
        '_model':     model,
        '_y_pred':    y_pred,
    }

results = []   # will hold all model results
print('Helper function ready!')


### Model 1 — Logistic Regression (Baseline)


In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    multi_class='auto',
    C=1.0,
    random_state=42,
    n_jobs=-1
)
res_lr = train_and_evaluate(lr, 'Logistic Regression', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_lr)


### Model 2 — Decision Tree


In [ ]:
dt = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
res_dt = train_and_evaluate(dt, 'Decision Tree', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_dt)


### Model 3 — Random Forest


In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)
res_rf = train_and_evaluate(rf, 'Random Forest', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_rf)


### Model 4 — XGBoost


In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)
res_xgb = train_and_evaluate(xgb, 'XGBoost', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_xgb)


### Model 5 — Voting Ensemble (RF + XGBoost)
Voting classifier takes predictions from both models and picks the majority vote.


In [ ]:
# Re-create fresh models for ensemble (untrained)
rf_v = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
xgb_v = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0)

voting = VotingClassifier(
    estimators=[
        ('random_forest', rf_v),
        ('xgboost',       xgb_v),
    ],
    voting='soft'     # 'soft' uses predicted probabilities — more accurate than 'hard'
)
res_voting = train_and_evaluate(voting, 'Voting Ensemble (RF+XGB)', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_voting)


### Model 6 — Stacking Ensemble
Base models: RF + XGBoost  
Meta-model: Logistic Regression  
The meta-model learns *how to combine* the base model predictions.


In [ ]:
rf_s  = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
xgb_s = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                       n_jobs=-1, random_state=42, eval_metric='mlogloss', verbosity=0)
meta  = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)

stacking = StackingClassifier(
    estimators=[
        ('random_forest', rf_s),
        ('xgboost',       xgb_s),
    ],
    final_estimator=meta,
    cv=3,              # 3-fold CV for generating meta-features
    n_jobs=-1,
    passthrough=False  # only pass predictions, not raw features, to meta-model
)
res_stacking = train_and_evaluate(stacking, 'Stacking Ensemble (RF+XGB→LR)', X_train_scaled, y_train_bal, X_test_scaled, y_test)
results.append(res_stacking)


## Step 4 — Model Comparison Table


In [ ]:
# Build comparison DataFrame
comparison_cols = ['Model', 'Accuracy', 'Macro F1', 'Weighted F1', 'Precision', 'Recall', 'Train Time']
results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
results_df = results_df[comparison_cols]

# Sort by Macro F1 descending
results_df_sorted = results_df.sort_values('Macro F1', ascending=False).reset_index(drop=True)
results_df_sorted.index += 1   # rank starts from 1

print('=' * 80)
print('  MODEL COMPARISON RESULTS')
print('=' * 80)
print(results_df_sorted.to_string())
print('=' * 80)
print(f'\nBest model: {results_df_sorted.iloc[0]["Model"]}  |  Macro F1: {results_df_sorted.iloc[0]["Macro F1"]}%')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models     = results_df_sorted['Model'].tolist()
accuracies = results_df_sorted['Accuracy'].tolist()
f1_scores  = results_df_sorted['Macro F1'].tolist()

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

# Accuracy chart
bars1 = axes[0].barh(models[::-1], accuracies[::-1], color=colors[:len(models)], alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Model Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlim(min(accuracies)-5, 102)
for bar, val in zip(bars1, accuracies[::-1]):
    axes[0].text(val+0.2, bar.get_y()+bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=9)

# F1 chart
bars2 = axes[1].barh(models[::-1], f1_scores[::-1], color=colors[:len(models)], alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Macro F1 Score (%)')
axes[1].set_title('Model Macro F1 Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlim(min(f1_scores)-5, 102)
for bar, val in zip(bars2, f1_scores[::-1]):
    axes[1].text(val+0.2, bar.get_y()+bar.get_height()/2, f'{val:.2f}%', va='center', fontsize=9)

plt.suptitle('All Models — Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')


In [ ]:
# Radar chart comparing all metrics across models
metrics = ['Accuracy', 'Macro F1', 'Precision', 'Recall']
N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c']

for i, row in results_df_sorted.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    color = colors_radar[(i-1) % len(colors_radar)]
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'], color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 105)
ax.set_yticks([20, 40, 60, 80, 100])
ax.set_yticklabels(['20%','40%','60%','80%','100%'], fontsize=8)
ax.set_title('Model Performance Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/model_radar.png', dpi=150, bbox_inches='tight')
plt.show()


## Step 5 — Cross Validation (Top 2 Models)
Cross validation gives a more reliable estimate of model performance.


In [ ]:
# Use a sample for speed (full CV on 2M rows takes very long)
sample_size = min(30000, len(X_train_scaled))
idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_cv = X_train_scaled[idx]
y_cv = y_train_bal[idx]

print(f'Running 5-fold cross-validation on {sample_size:,} samples...')
print('(Sampled for speed — full dataset would take ~30 min)')
print()

cv_results = {}

for model_obj, name in [(rf, 'Random Forest'), (xgb, 'XGBoost')]:
    t0 = time.time()
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model_obj, X_cv, y_cv, cv=skf, scoring='f1_macro', n_jobs=-1)
    elapsed = time.time() - t0
    cv_results[name] = scores
    print(f'{name}:')
    print(f'  Fold scores : {[round(s,4) for s in scores]}')
    print(f'  Mean ± Std  : {scores.mean():.4f} ± {scores.std():.4f}')
    print(f'  Time        : {elapsed:.1f}s')
    print()


## Step 6 — Confusion Matrix (Best Model)


In [ ]:
# Get best model by Macro F1
best_row   = results_df_sorted.iloc[0]
best_name  = best_row['Model']
best_result = next(r for r in results if r['Model'] == best_name)
best_model  = best_result['_model']
best_y_pred = best_result['_y_pred']

print(f'Best model: {best_name}')

# Confusion matrix
cm = confusion_matrix(y_test, best_y_pred)
fig, ax = plt.subplots(figsize=(14, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrix_best.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
print(f'Classification Report — {best_name}')
print('=' * 70)
print(classification_report(y_test, best_y_pred, target_names=le.classes_, zero_division=0))


## Step 7 — Feature Importance (XGBoost)


In [ ]:
feature_names = list(X.columns)

# XGBoost feature importance
xgb_importances = xgb.feature_importances_
top_n = 20
indices = np.argsort(xgb_importances)[::-1][:top_n]
top_features_names  = [feature_names[i] for i in indices]
top_features_scores = xgb_importances[indices]

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_features_scores[::-1], color='#3498db', alpha=0.8)
plt.yticks(range(top_n), top_features_names[::-1], fontsize=9)
plt.xlabel('Importance Score')
plt.title(f'XGBoost — Top {top_n} Feature Importances', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most important features:')
for i in range(min(10, top_n)):
    print(f'  {i+1:2d}. {top_features_names[i]:<40}  {top_features_scores[i]:.4f}')


## Step 8 — Save Best Model


In [ ]:
# Save best model
joblib.dump(best_model, '../model.pkl')

print('=' * 55)
print('  TRAINING COMPLETE')
print('=' * 55)
print(f'  Best model     : {best_name}')
print(f'  Accuracy       : {best_row["Accuracy"]}%')
print(f'  Macro F1       : {best_row["Macro F1"]}%')
print(f'  Weighted F1    : {best_row["Weighted F1"]}%')
print('  Saved files:')
print('    ../model.pkl')
print('    ../scaler.pkl')
print('    ../label_encoder.pkl')
print('=' * 55)

print('\nAll model results summary:')
print(results_df_sorted[['Model','Accuracy','Macro F1','Weighted F1']].to_string(index=True))


## Final Summary

### What we did:
1. Loaded and cleaned CICIDS2017 dataset
2. Applied SMOTE to fix class imbalance
3. Scaled features with StandardScaler
4. Trained **4 individual models**: Logistic Regression, Decision Tree, Random Forest, XGBoost
5. Built **2 ensemble models**: Voting (RF+XGB) and Stacking (RF+XGB → LR)
6. Compared all 6 models on Accuracy, F1, Precision, Recall
7. Ran 5-fold cross-validation on top 2 models
8. Plotted confusion matrix and feature importance
9. Saved best model to `model.pkl`

### Key Takeaway:
- **Tree-based models (RF, XGBoost) dominate** on network flow tabular data
- **Ensemble methods improve** slightly over individual models
- **Logistic Regression is weakest** but trains fastest
- **Decision Tree overfits** slightly — RF fixes this with bagging

### Next step:
The saved `model.pkl` is loaded by `src/model/predict.py` which is called by the FastAPI backend.
